#### 构建label
用一周周末的收益率变化作为label
* 基金用的参考odf_funds
* 指数参考沪深300（000300.SH）
* 无风险利率用的是一年期国债未到期存款利率

In [39]:
import pandas as pd
from config.config import *
import numpy as np

In [40]:
# df = pd.read_feather('data/funds_with_scale.feather')
df = pd.read_feather('data/funds_with_scale_stock.feather')
df

nav_date  accum_nav    adj_nav     fd_share
ann_date   ts_code                                               
2016-03-30 000011.OF  20160329    13.6370  16.694700   18642.5278
           000309.OF  20160329     1.4450   1.445000   32296.7705
           000409.OF  20160329     1.3490   1.349000   36018.4498
           000471.OF  20160329     2.2190   2.219000  177241.4719
           000513.OF  20160329     1.3610   1.361000   74609.3099
...                        ...        ...        ...          ...
2026-03-31 540008.OF  20260330     2.9249   3.079717  155482.9886
           540009.OF  20260330     1.7303   1.714082   24682.4579
           540010.OF  20260330     4.1115   4.111500   16997.8038
           550008.OF  20260330     2.9682   3.523751  132721.6502
           960000.OF  20260330     2.2753   2.275300    3922.2942

[494930 rows x 4 columns]

In [41]:
import datetime
index_df = pd.read_csv('index/000300.SH.csv')
index_df['trade_date'] = pd.to_datetime(index_df['trade_date'])
index_df.set_index(['trade_date'], inplace=True)
index_df.sort_index(inplace=True)


index_df = index_df.loc[index_df.index.get_level_values("trade_date") >= "2016-03-30"]
index_df

,close
trade_date,
2016-03-30,3216.2753
2016-03-31,3218.0879
2016-04-01,3221.8948
2016-04-05,3264.4858
2016-04-06,3257.5276
...,...
2026-03-23,4417.9971
2026-03-24,4474.7224
2026-03-25,4537.4656


In [42]:
index_month = index_df.resample('M').last()
index_ret = index_month.pct_change().dropna()
index_ret

/var/folders/ss/hq0_l7mj2f35ct8zlh27s1p80000gn/T/ipykernel_57162/343495864.py:1: FutureWarning: 'M' is deprecated and will be removed in a future version, please use 'ME' instead.
  index_month = index_df.resample('M').last()


,close
trade_date,
2016-04-30,-0.019062
2016-05-31,0.004059
2016-06-30,-0.004934
2016-07-31,0.015856
2016-08-31,0.038660
...,...
2025-11-30,-0.024567
2025-12-31,0.022815
2026-01-31,0.016501


构造：
* label1: 全部基金等权
* label2: 全部基金fd_share加权
* label3: 前25%基金等权
* label4: 前25%基金fd_share加权

In [43]:
df_month = df.groupby('ts_code').resample('M', level='ann_date').last()
df_month['label'] = df_month.groupby('ts_code')['adj_nav'].pct_change()
df_month = df_month.dropna(subset=['label'])
df_month = df_month.swaplevel()
df_month.sort_index(inplace=True)
df_month

/var/folders/ss/hq0_l7mj2f35ct8zlh27s1p80000gn/T/ipykernel_57162/57529793.py:1: FutureWarning: 'M' is deprecated and will be removed in a future version, please use 'ME' instead.
  df_month = df.groupby('ts_code').resample('M', level='ann_date').last()


nav_date  accum_nav    adj_nav     fd_share     label
ann_date   ts_code                                                         
2016-04-30 000011.OF  20160429    13.9370  17.213300   18642.5278  0.002014
           000309.OF  20160429     1.5390   1.539000   32296.7705  0.035666
           000409.OF  20160429     1.4010   1.401000   36018.4498  0.007914
           000471.OF  20160429     2.2620   2.262000  177241.4719 -0.013089
           000513.OF  20160429     1.3900   1.390000   74609.3099 -0.024561
...                        ...        ...        ...          ...       ...
2026-03-31 540008.OF  20260330     2.9249   3.079717  155482.9886 -0.069103
           540009.OF  20260330     1.7303   1.714082   24682.4579 -0.082504
           540010.OF  20260330     4.1115   4.111500   16997.8038  0.015913
           550008.OF  20260330     2.9682   3.523751  132721.6502 -0.050345
           960000.OF  20260330     2.2753   2.275300    3922.2942 -0.056049

[21716 rows x 5 columns]

In [44]:
# 构造加权label
df_month['total_fd_share'] = df_month.groupby('ann_date')['fd_share'].transform('sum')
df_month['weight'] = df_month['fd_share'] / df_month['total_fd_share']
df_month = df_month.drop(columns=['total_fd_share'])
df_month 

nav_date  accum_nav    adj_nav     fd_share     label  \
ann_date   ts_code                                                            
2016-04-30 000011.OF  20160429    13.9370  17.213300   18642.5278  0.002014   
           000309.OF  20160429     1.5390   1.539000   32296.7705  0.035666   
           000409.OF  20160429     1.4010   1.401000   36018.4498  0.007914   
           000471.OF  20160429     2.2620   2.262000  177241.4719 -0.013089   
           000513.OF  20160429     1.3900   1.390000   74609.3099 -0.024561   
...                        ...        ...        ...          ...       ...   
2026-03-31 540008.OF  20260330     2.9249   3.079717  155482.9886 -0.069103   
           540009.OF  20260330     1.7303   1.714082   24682.4579 -0.082504   
           540010.OF  20260330     4.1115   4.111500   16997.8038  0.015913   
           550008.OF  20260330     2.9682   3.523751  132721.6502 -0.050345   
           960000.OF  20260330     2.2753   2.275300    3922.2942 -0.056049   

                        weight  
ann_date   ts_code              
2016-04-30 000011.OF  0.000767  
           000309.OF  0.001328  
           000409.OF  0.001481  
           000471.OF  0.007290  
           000513.OF  0.003069  
...                        ...  
2026-03-31 540008.OF  0.013204  
           540009.OF  0.002096  
           540010.OF  0.001444  
           550008.OF  0.011271  
           960000.OF  0.000333  

[21716 rows x 6 columns]

In [45]:
rf0 = pd.read_csv('index/3m_deposit_rate.csv')
rf = rf0[['trade_date','yield']]
rf['trade_date'] = pd.to_datetime(rf['trade_date'],format ="%Y-%m-%d")
rf.set_index(['trade_date'],inplace=True)
rf.sort_index()
rf

,yield
trade_date,
1990-04-15,0.063
1990-04-16,0.063
1990-04-17,0.063
1990-04-18,0.063
1990-04-19,0.063
...,...
2026-03-23,0.011
2026-03-24,0.011
2026-03-25,0.011


In [46]:
rf = rf.loc[rf.index.get_level_values("trade_date") >= "2016-02-01"]
rf = rf.resample('M').last()
rf_month = rf
# rf_month = rf.pct_change().dropna()
rf_month

/var/folders/ss/hq0_l7mj2f35ct8zlh27s1p80000gn/T/ipykernel_57162/3637534972.py:2: FutureWarning: 'M' is deprecated and will be removed in a future version, please use 'ME' instead.
  rf = rf.resample('M').last()


,yield
trade_date,
2016-02-29,0.011
2016-03-31,0.011
2016-04-30,0.011
2016-05-31,0.011
2016-06-30,0.011
...,...
2025-11-30,0.011
2025-12-31,0.011
2026-01-31,0.011


In [47]:
index_ret = index_ret.merge(rf_month, on='trade_date')
index_ret['excess'] = index_ret['close'] - index_ret['yield']
index_ret

,close,yield,excess
trade_date,,,
2016-04-30,-0.019062,0.011,-0.030062
2016-05-31,0.004059,0.011,-0.006941
2016-06-30,-0.004934,0.011,-0.015934
2016-07-31,0.015856,0.011,0.004856
2016-08-31,0.038660,0.011,0.027660
...,...,...,...
2025-11-30,-0.024567,0.011,-0.035567
2025-12-31,0.022815,0.011,0.011815
2026-01-31,0.016501,0.011,0.005501


In [48]:
df_month.to_feather('data/month_label.feather')
index_ret.to_feather('data/month_HS300_rf.feather')